# Laboratory Activity 6 — CNN Implementation

**Instruction from the uploaded notebook:** Convert the given CNN architecture diagram into a PyTorch CNN Architecture.

This solution follows the **attached diagram**, including its specified kernel sizes, strides, padding, dropout, and fully connected layers.


## Architecture from the Diagram

Input image: **1 × 28 × 28**

For a batch size of 32:

- Conv1: 1 → 32 channels, kernel 3×3, stride 1, padding 1
- ReLU
- MaxPool1: kernel 2×2, stride 1, padding 1
- Conv2: 32 → 64 channels, kernel 3×3, stride 1, padding 1
- ReLU
- Conv3: 64 → 128 channels, kernel 3×3, stride 1, padding 1
- ReLU
- Conv4: 128 → 256 channels, kernel 3×3, stride 1, padding 1
- ReLU
- MaxPool2: kernel 2×2, stride 1, padding 0
- Dropout: p = 0.2
- Flatten
- FCN1: flattened features → 1000
- ReLU
- FCN2: 1000 → 500
- ReLU
- FCN3: 500 → 10
- Softmax

The notebook being used for this activity works with **10 output classes**, so FCN3 uses 10 outputs.


## 1. Import PyTorch Libraries


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


## 2. Determine the Feature-Map Shapes

The output size is calculated using:

**Output = floor((Input + 2P − K) / S) + 1**

where:
- `P` = padding
- `K` = kernel size
- `S` = stride


In [2]:
# Spatial dimensions based exactly on the attached architecture

# Input: 28 x 28
# Conv1: kernel=3, stride=1, padding=1
conv1_size = ((28 + 2*1 - 3) // 1) + 1

# MaxPool1: kernel=2, stride=1, padding=1
pool1_size = ((conv1_size + 2*1 - 2) // 1) + 1

# Conv2, Conv3, Conv4 all preserve spatial size
conv2_size = ((pool1_size + 2*1 - 3) // 1) + 1
conv3_size = ((conv2_size + 2*1 - 3) // 1) + 1
conv4_size = ((conv3_size + 2*1 - 3) // 1) + 1

# MaxPool2: kernel=2, stride=1, padding=0
pool2_size = ((conv4_size + 2*0 - 2) // 1) + 1

flatten_features = 256 * pool2_size * pool2_size

print("After Conv1   :", (32, 32, conv1_size, conv1_size))
print("After MaxPool1:", (32, 32, pool1_size, pool1_size))
print("After Conv2   :", (32, 64, conv2_size, conv2_size))
print("After Conv3   :", (32, 128, conv3_size, conv3_size))
print("After Conv4   :", (32, 256, conv4_size, conv4_size))
print("After MaxPool2:", (32, 256, pool2_size, pool2_size))
print("Flatten size  :", flatten_features)


After Conv1   : (32, 32, 28, 28)
After MaxPool1: (32, 32, 29, 29)
After Conv2   : (32, 64, 29, 29)
After Conv3   : (32, 128, 29, 29)
After Conv4   : (32, 256, 29, 29)
After MaxPool2: (32, 256, 28, 28)
Flatten size  : 200704


## 3. Create the CNN Architecture


In [3]:
class CNNLab6(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=(3, 3),
            stride=1,
            padding=1
        )

        self.pool1 = nn.MaxPool2d(
            kernel_size=(2, 2),
            stride=1,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=(3, 3),
            stride=1,
            padding=1
        )

        self.conv3 = nn.Conv2d(
            in_channels=64,
            out_channels=128,
            kernel_size=(3, 3),
            stride=1,
            padding=1
        )

        self.conv4 = nn.Conv2d(
            in_channels=128,
            out_channels=256,
            kernel_size=(3, 3),
            stride=1,
            padding=1
        )

        self.pool2 = nn.MaxPool2d(
            kernel_size=(2, 2),
            stride=1,
            padding=0
        )

        # Dropout specified in the diagram
        self.dropout = nn.Dropout(p=0.2)

        # After MaxPool2: 256 x 28 x 28
        self.fcn1 = nn.Linear(256 * 28 * 28, 1000)
        self.fcn2 = nn.Linear(1000, 500)
        self.fcn3 = nn.Linear(500, 10)

    def forward(self, x):
        # Conv1 -> ReLU -> MaxPool1
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        # Conv2 -> ReLU
        x = F.relu(self.conv2(x))

        # Conv3 -> ReLU
        x = F.relu(self.conv3(x))

        # Conv4 -> ReLU -> MaxPool2
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        # Dropout
        x = self.dropout(x)

        # Flatten
        x = torch.flatten(x, 1)

        # Fully connected layers
        x = F.relu(self.fcn1(x))
        x = F.relu(self.fcn2(x))
        x = self.fcn3(x)

        # Softmax output
        x = F.softmax(x, dim=1)

        return x


## 4. Instantiate the Model


In [4]:
model = CNNLab6()
print(model)


CNNLab6(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=1, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=(2, 2), stride=1, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fcn1): Linear(in_features=200704, out_features=1000, bias=True)
  (fcn2): Linear(in_features=1000, out_features=500, bias=True)
  (fcn3): Linear(in_features=500, out_features=10, bias=True)
)


## 5. Verify the Architecture Using a Dummy Batch

The diagram shows a batch dimension of **32**, so a dummy input with shape `(32, 1, 28, 28)` is used to verify the model.


In [5]:
dummy_input = torch.randn(32, 1, 28, 28)

model.eval()
with torch.no_grad():
    output = model(dummy_input)

print("Input shape :", dummy_input.shape)
print("Output shape:", output.shape)
print("Expected output shape: torch.Size([32, 10])")


Input shape : torch.Size([32, 1, 28, 28])
Output shape: torch.Size([32, 10])
Expected output shape: torch.Size([32, 10])


## 6. Verify Each Layer's Shape


In [6]:
x = dummy_input

with torch.no_grad():
    x = F.relu(model.conv1(x))
    print("Conv1     :", x.shape)

    x = model.pool1(x)
    print("MaxPool1  :", x.shape)

    x = F.relu(model.conv2(x))
    print("Conv2     :", x.shape)

    x = F.relu(model.conv3(x))
    print("Conv3     :", x.shape)

    x = F.relu(model.conv4(x))
    print("Conv4     :", x.shape)

    x = model.pool2(x)
    print("MaxPool2  :", x.shape)

    x = model.dropout(x)
    print("Dropout   :", x.shape)

    x = torch.flatten(x, 1)
    print("Flatten   :", x.shape)

    x = F.relu(model.fcn1(x))
    print("FCN1      :", x.shape)

    x = F.relu(model.fcn2(x))
    print("FCN2      :", x.shape)

    x = F.softmax(model.fcn3(x), dim=1)
    print("FCN3      :", x.shape)


Conv1     : torch.Size([32, 32, 28, 28])
MaxPool1  : torch.Size([32, 32, 29, 29])
Conv2     : torch.Size([32, 64, 29, 29])
Conv3     : torch.Size([32, 128, 29, 29])
Conv4     : torch.Size([32, 256, 29, 29])


MaxPool2  : torch.Size([32, 256, 28, 28])
Dropout   : torch.Size([32, 256, 28, 28])
Flatten   : torch.Size([32, 200704])
FCN1      : torch.Size([32, 1000])
FCN2      : torch.Size([32, 500])
FCN3      : torch.Size([32, 10])


## Final Architecture Shapes

| Layer | Output Shape for Batch Size 32 |
|---|---|
| Input | `(32, 1, 28, 28)` |
| Conv1 | `(32, 32, 28, 28)` |
| MaxPool1 | `(32, 32, 29, 29)` |
| Conv2 | `(32, 64, 29, 29)` |
| Conv3 | `(32, 128, 29, 29)` |
| Conv4 | `(32, 256, 29, 29)` |
| MaxPool2 | `(32, 256, 28, 28)` |
| Dropout | `(32, 256, 28, 28)` |
| Flatten | `(32, 200704)` |
| FCN1 | `(32, 1000)` |
| FCN2 | `(32, 500)` |
| FCN3 / Softmax | `(32, 10)` |

**Important:** MaxPool1 produces `29 × 29`, not `14 × 14`, because the attached diagram specifically gives `stride=1` and `padding=1`. This implementation follows the diagram exactly.
